# AGML 5-Fold Training Notebook

This notebook is **self-contained for training**.

The only external script you need is the dataset fold generator:
- `make_5fold_dataset.py`

That script should create:

```text
data_5_fold/
  fold_1/
    train/
      normal/
      subluxation/
    validation/
      normal/
      subluxation/
    test/
      normal/
      subluxation/
  ...
  fold_5/
```

After that, this notebook handles:
- model definition
- artifact augmentation during training
- one-fold testing
- sequential 5-fold training
- result aggregation

## Important
- Do **not** pre-generate artifact images into the fold folders.
- Use the clean fold folders as the source dataset.
- Artifact corruption is applied in the loader during training/evaluation when `scenario="artifact_mix"`.


In [ ]:
from __future__ import annotations

import hashlib
import math
import os
import sys
import zipfile
import random
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import csv
import gc
import hashlib
import importlib.util
import json
import shutil
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold, train_test_split
from tensorflow.keras import Model
from tensorflow.keras import layers
from tensorflow.keras import mixed_precision
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import Sequence as KerasSequence, to_categorical

In [ ]:
# CUDA Log Level: 0 = all messages are logged (default behavior), 
# 1 = INFO messages are not printed, 
# 2 = INFO and WARNING messages are not printed, 
# 3 = INFO, WARNING, and ERROR messages are not printed
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

# Disable TensorFlow's JIT compiler (XLA) to prevent potential issues with mixed precision
tf.config.optimizer.set_jit(False)
mixed_precision.set_global_policy("mixed_float16")

## Colab / Local Dataset Setup

Use this section before training.

### Recommended Colab workflow for a heavy dataset

1. Upload your zip to Google Drive, for example:
   `MyDrive/AGML/data_5_fold.zip`

2. Mount Drive in this notebook.

3. Set `ZIP_PATH` to the zip file path.

4. Run the extraction cell.

The zip may contain either:

```text
data_5_fold/
  fold_1/
  fold_2/
  fold_3/
  fold_4/
  fold_5/
```

or directly:

```text
fold_1/
fold_2/
fold_3/
fold_4/
fold_5/
```

If your zip contains only the original `data/train`, `data/validation`, and `data/test` folders, run the separate `make_5fold_dataset.py` first, then zip the resulting `data_5_fold/` folder for Colab training.


In [ ]:
# Colab / local runtime detection and zip extraction

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive, files
    BASE_DIR = Path("/content")
    print("Running in Google Colab.")
else:
    BASE_DIR = Path(".").resolve()
    print("Running locally.")

# Change this if you want a different extraction location.
EXTRACT_ROOT = BASE_DIR / "agml_dataset"

# Option A: Google Drive path. Recommended for heavy zip files.
# Example after mounting Drive:
# ZIP_PATH = Path("/content/drive/MyDrive/AGML/data_5_fold.zip")

# Option B: direct Colab upload. Use only for smaller zips.
# ZIP_PATH = Path("/content/data_5_fold.zip")

ZIP_PATH = None  # Set this to a Path before running extract_dataset_zip(...)


def mount_google_drive() -> None:
    """Mount Google Drive in Colab. Does nothing locally."""
    if not IN_COLAB:
        print("Not in Colab; skipping Drive mount.")
        return
    drive.mount("/content/drive")


def upload_zip_to_colab() -> Path:
    """
    Browser upload helper for Colab.
    Not recommended for very large zip files; Google Drive is more stable.
    """
    if not IN_COLAB:
        raise RuntimeError("upload_zip_to_colab() only works in Google Colab.")

    uploaded = files.upload()
    zip_candidates = [name for name in uploaded.keys() if name.lower().endswith(".zip")]
    if not zip_candidates:
        raise FileNotFoundError("No .zip file was uploaded.")

    zip_path = BASE_DIR / zip_candidates[0]
    print("Uploaded zip:", zip_path)
    return zip_path


def extract_dataset_zip(zip_path: Path, extract_root: Path = EXTRACT_ROOT, overwrite: bool = False) -> Path:
    """
    Extract a dataset zip into extract_root.

    If overwrite=True, the existing extract_root is deleted first.
    """
    zip_path = Path(zip_path)

    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")

    if overwrite and extract_root.exists():
        shutil.rmtree(extract_root)

    extract_root.mkdir(parents=True, exist_ok=True)

    print(f"Extracting: {zip_path}")
    print(f"To:         {extract_root}")

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_root)

    print("Extraction complete.")
    return extract_root


def has_fold_layout(path: Path) -> bool:
    """Return True if path contains fold_1 ... fold_5."""
    path = Path(path)
    return all((path / f"fold_{i}").exists() for i in range(1, 6))


def find_folds_root(search_root: Path) -> Path:
    """
    Auto-detect the folder that contains fold_1 ... fold_5.

    Supports:
    - extract_root/data_5_fold/fold_1...
    - extract_root/fold_1...
    - extract_root/some_nested_folder/data_5_fold/fold_1...
    """
    search_root = Path(search_root)

    if has_fold_layout(search_root):
        return search_root

    direct = search_root / "data_5_fold"
    if has_fold_layout(direct):
        return direct

    candidates = []
    for candidate in search_root.rglob("*"):
        if candidate.is_dir() and has_fold_layout(candidate):
            candidates.append(candidate)

    if not candidates:
        raise FileNotFoundError(
            "Could not find a folder containing fold_1 ... fold_5. "
            "Make sure your zip contains data_5_fold/fold_1 ... fold_5."
        )

    # Prefer a folder named data_5_fold if available.
    candidates = sorted(candidates, key=lambda p: (p.name != "data_5_fold", len(str(p))))
    return candidates[0]


def summarize_folds_root(folds_root: Path) -> pd.DataFrame:
    """Quick count check for a detected data_5_fold folder."""
    rows = []
    for fold_idx in range(1, 6):
        fold_dir = Path(folds_root) / f"fold_{fold_idx}"
        for split_name in ["train", "validation", "test"]:
            for class_name in ["normal", "subluxation"]:
                class_dir = fold_dir / split_name / class_name
                count = 0
                if class_dir.exists():
                    count = sum(
                        1 for p in class_dir.rglob("*")
                        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
                    )
                rows.append({
                    "fold": f"fold_{fold_idx}",
                    "split": split_name,
                    "class_name": class_name,
                    "count": count,
                })
    return pd.DataFrame(rows)


### Choose one Colab dataset option

#### Option 1: Zip is already in Google Drive

Run:

```python
mount_google_drive()
ZIP_PATH = Path("/content/drive/MyDrive/AGML/data_5_fold.zip")
extract_dataset_zip(ZIP_PATH, EXTRACT_ROOT, overwrite=True)
FOLDS_ROOT = find_folds_root(EXTRACT_ROOT)
summarize_folds_root(FOLDS_ROOT)
```

#### Option 2: Upload zip directly in Colab

Run:

```python
ZIP_PATH = upload_zip_to_colab()
extract_dataset_zip(ZIP_PATH, EXTRACT_ROOT, overwrite=True)
FOLDS_ROOT = find_folds_root(EXTRACT_ROOT)
summarize_folds_root(FOLDS_ROOT)
```

#### Option 3: Local laptop

Set:

```python
FOLDS_ROOT = Path("data_5_fold")
summarize_folds_root(FOLDS_ROOT)
```


In [ ]:
# Pick ONE option below.

# ===== Option 1: Google Drive zip, recommended for Colab =====
# mount_google_drive()
# ZIP_PATH = Path("/content/drive/MyDrive/AGML/data_5_fold.zip")
# extract_dataset_zip(ZIP_PATH, EXTRACT_ROOT, overwrite=True)
# FOLDS_ROOT = find_folds_root(EXTRACT_ROOT)

# ===== Option 2: Browser upload zip in Colab =====
# ZIP_PATH = upload_zip_to_colab()
# extract_dataset_zip(ZIP_PATH, EXTRACT_ROOT, overwrite=True)
# FOLDS_ROOT = find_folds_root(EXTRACT_ROOT)

# ===== Option 3: Local machine, already extracted =====
FOLDS_ROOT = Path("data_5_fold")

print("FOLDS_ROOT =", Path(FOLDS_ROOT).resolve())

# Run this after choosing your option.
summarize_folds_root(FOLDS_ROOT)


In [48]:
# Reproducibility

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

In [49]:
# Labels and config

TMD_LABELS = ["normal", "subluxation"]
DISPLAY_LABELS = ["Normal", "Subluxation"]
CLASS_NAME_ALIASES = {
    "normal": 0,
    "subluxation": 1,
    "sl": 1,
}
ARTIFACT_LABELS = ["none", "motion_blur", "gaussian_noise", "metal_streak"]
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

@dataclass
class ExperimentConfig:
    dataset_root: str
    image_size: Tuple[int, int] = (224, 224)
    batch_size: int = 8
    epochs: int = 30
    learning_rate: float = 1e-4
    weight_decay: float = 1e-2
    n_splits: int = 5
    val_size: float = 0.15
    random_state: int = 42
    freeze_backbone: bool = False
    benchmark_attention: str = "self"  # corrected baseline attention
    proposed_attention: str = "cbam"
    artifact_mix_probability: Tuple[float, float, float, float] = (0.25, 0.25, 0.25, 0.25)
    tmd_loss_weight: float = 1.0
    artifact_loss_weight: float = 0.35
    num_workers: int = 1
    max_queue_size: int = 10

In [50]:
# Dataset indexing

def index_dataset(dataset_root: str) -> pd.DataFrame:
    root = Path(dataset_root)
    if not root.exists():
        raise FileNotFoundError(f"Dataset root does not exist: {dataset_root}")

    rows: List[Dict[str, object]] = []
    class_to_idx = {name: idx for idx, name in enumerate(TMD_LABELS)}
    split_names = ["train", "validation", "test"]

    # Case 1: fixed split layout
    has_split_layout = all((root / split_name).exists() for split_name in split_names)

    if has_split_layout:
        for split_name in split_names:
            split_dir = root / split_name
            for class_name in TMD_LABELS:
                class_dir = split_dir / class_name
                if not class_dir.exists():
                    raise FileNotFoundError(
                        f"Expected class folder '{class_name}' inside {split_dir}"
                    )

                for file_path in class_dir.rglob("*"):
                    if file_path.is_file() and file_path.suffix.lower() in IMAGE_EXTENSIONS:
                        rows.append(
                            {
                                "filepath": str(file_path),
                                "tmd_label": class_to_idx[class_name],
                                "class_name": class_name,
                                "split": split_name,
                            }
                        )
    else:
        # Case 2: flat layout for cross-validation
        for class_name in TMD_LABELS:
            class_dir = root / class_name
            if not class_dir.exists():
                raise FileNotFoundError(
                    f"Expected class folder '{class_name}' inside {dataset_root}"
                )
            for file_path in class_dir.rglob("*"):
                if file_path.is_file() and file_path.suffix.lower() in IMAGE_EXTENSIONS:
                    rows.append(
                        {
                            "filepath": str(file_path),
                            "tmd_label": class_to_idx[class_name],
                            "class_name": class_name,
                            "split": "all",
                        }
                    )

    if not rows:
        raise ValueError(
            "No image files found. Expected either:\n"
            "1) dataset_root/train|validation|test/normal|subluxation\n"
            "or\n"
            "2) dataset_root/normal|subluxation"
        )

    df = pd.DataFrame(rows).sort_values("filepath").reset_index(drop=True)
    return df


In [51]:
# Image corruption utilities

def _ensure_uint8(image: np.ndarray) -> np.ndarray:
    image = np.clip(image, 0, 255)
    return image.astype(np.uint8)


def add_motion_blur(image: np.ndarray, kernel_size: int = 9) -> np.ndarray:
    kernel_size = max(3, int(kernel_size))
    if kernel_size % 2 == 0:
        kernel_size += 1
    kernel = np.zeros((kernel_size, kernel_size), dtype=np.float32)
    kernel[kernel_size // 2, :] = 1.0
    kernel /= kernel.sum()
    blurred = cv2.filter2D(image, -1, kernel)
    return _ensure_uint8(blurred)


def add_gaussian_noise(image: np.ndarray, sigma: float = 18.0) -> np.ndarray:
    noise = np.random.normal(0, sigma, image.shape)
    noisy = image.astype(np.float32) + noise
    return _ensure_uint8(noisy)


def add_metal_streak(image: np.ndarray, num_streaks: int = 3) -> np.ndarray:
    h, w = image.shape[:2]
    output = image.copy().astype(np.float32)

    for _ in range(max(1, num_streaks)):
        x1 = np.random.randint(0, w)
        y1 = np.random.randint(0, h)
        x2 = np.random.randint(0, w)
        y2 = np.random.randint(0, h)
        thickness = np.random.randint(2, 8)

        overlay = np.zeros_like(output)
        intensity = np.random.randint(180, 255)
        cv2.line(overlay, (x1, y1), (x2, y2), (intensity, intensity, intensity), thickness)
        overlay = cv2.GaussianBlur(overlay, (0, 0), sigmaX=2.0)
        output = np.maximum(output, overlay)

    return _ensure_uint8(output)


def apply_artifact(image: np.ndarray, artifact_label: int) -> np.ndarray:
    if artifact_label == 0:
        return image
    if artifact_label == 1:
        return add_motion_blur(image)
    if artifact_label == 2:
        return add_gaussian_noise(image)
    if artifact_label == 3:
        return add_metal_streak(image)
    raise ValueError(f"Unknown artifact label: {artifact_label}")

In [52]:
# Data generator

class TMJSequence(KerasSequence):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        image_size: Tuple[int, int],
        batch_size: int,
        multi_task: bool,
        scenario: str,
        training: bool,
        seed: int = 42,
    ) -> None:
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_size = image_size
        self.batch_size = batch_size
        self.multi_task = multi_task
        self.scenario = scenario
        self.training = training
        self.seed = seed
        self.indices = np.arange(len(self.df))
        self.rng = np.random.default_rng(seed)
        self.on_epoch_end()

    def __len__(self) -> int:
        return math.ceil(len(self.df) / self.batch_size)

    def on_epoch_end(self) -> None:
        if self.training:
            self.rng.shuffle(self.indices)

    def _deterministic_artifact(self, filepath: str) -> int:
        if self.scenario == "clean":
            return 0
        digest = hashlib.md5(filepath.encode("utf-8")).hexdigest()
        return int(digest, 16) % len(ARTIFACT_LABELS)

    def _sample_artifact(self, filepath: str) -> int:
        if self.scenario == "clean":
            return 0
        if self.training:
            return int(self.rng.choice(np.arange(len(ARTIFACT_LABELS))))
        return self._deterministic_artifact(filepath)

    def _load_image(self, filepath: str) -> np.ndarray:
        image = cv2.imread(filepath)
        if image is None:
            raise ValueError(f"Failed to load image: {filepath}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, self.image_size, interpolation=cv2.INTER_AREA)
        return image

    def _augment_base(self, image: np.ndarray) -> np.ndarray:
        if self.training and self.rng.random() < 0.5:
            image = cv2.flip(image, 1)
        return image

    def __getitem__(self, index: int):
        batch_indices = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        batch_df = self.df.iloc[batch_indices]

        images: List[np.ndarray] = []
        tmd_labels: List[int] = []
        artifact_labels: List[int] = []

        for row in batch_df.itertuples(index=False):
            image = self._load_image(row.filepath)
            image = self._augment_base(image)
            artifact_label = self._sample_artifact(row.filepath)
            image = apply_artifact(image, artifact_label)
            image = image.astype(np.float32) / 255.0

            images.append(image)
            tmd_labels.append(int(row.tmd_label))
            artifact_labels.append(int(artifact_label))

        x = np.stack(images, axis=0)
        y_tmd = to_categorical(np.array(tmd_labels), num_classes=len(TMD_LABELS))
        y_artifact = to_categorical(np.array(artifact_labels), num_classes=len(ARTIFACT_LABELS))

        if self.multi_task:
            return x, {"tmd_output": y_tmd, "artifact_output": y_artifact}
        return x, y_tmd

In [53]:
# Attention blocks

class AttentionBlock(layers.Layer):
    def __init__(self, attention_type: str = "cbam", reduction_ratio: int = 16, **kwargs):
        super().__init__(**kwargs)
        self.attention_type = attention_type.lower()
        self.reduction_ratio = reduction_ratio

    def build(self, input_shape):
        filters = int(input_shape[-1])
        reduced_filters = max(filters // self.reduction_ratio, 1)

        if self.attention_type == "self":
            qk_filters = max(filters // 8, 1)
            self.query_conv = layers.Conv2D(qk_filters, 1, padding="same")
            self.key_conv = layers.Conv2D(qk_filters, 1, padding="same")
            self.value_conv = layers.Conv2D(filters, 1, padding="same")
        elif self.attention_type == "cbam":
            self.avg_pool = layers.GlobalAveragePooling2D()
            self.max_pool = layers.GlobalMaxPooling2D()
            self.shared_dense_1 = layers.Dense(reduced_filters, activation="relu")
            self.shared_dense_2 = layers.Dense(filters)
            self.spatial_conv = layers.Conv2D(1, 7, padding="same", activation="sigmoid")
        else:
            raise ValueError(f"Unsupported attention type: {self.attention_type}")

        super().build(input_shape)

    def call(self, inputs):
        if self.attention_type == "self":
            shape = tf.shape(inputs)
            batch_size, height, width = shape[0], shape[1], shape[2]
            channels = inputs.shape[-1]

            q = self.query_conv(inputs)
            k = self.key_conv(inputs)
            v = self.value_conv(inputs)

            q = tf.reshape(q, [batch_size, height * width, q.shape[-1]])
            k = tf.reshape(k, [batch_size, height * width, k.shape[-1]])
            v = tf.reshape(v, [batch_size, height * width, channels])

            attention_scores = tf.matmul(q, k, transpose_b=True)
            attention_scores = attention_scores / tf.math.sqrt(tf.cast(tf.shape(k)[-1], tf.float32))
            attention_weights = tf.nn.softmax(attention_scores, axis=-1)
            attended = tf.matmul(attention_weights, v)
            attended = tf.reshape(attended, [batch_size, height, width, channels])
            return inputs + attended

        avg_descriptor = self.avg_pool(inputs)
        max_descriptor = self.max_pool(inputs)
        avg_descriptor = self.shared_dense_2(self.shared_dense_1(avg_descriptor))
        max_descriptor = self.shared_dense_2(self.shared_dense_1(max_descriptor))
        channel_attention = tf.nn.sigmoid(avg_descriptor + max_descriptor)
        channel_attention = tf.reshape(channel_attention, (-1, 1, 1, inputs.shape[-1]))
        x = inputs * channel_attention

        avg_map = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_map = tf.reduce_max(x, axis=-1, keepdims=True)
        spatial_attention = self.spatial_conv(tf.concat([avg_map, max_map], axis=-1))
        return x * spatial_attention

In [54]:
# Model builders

def _make_backbone(image_size: Tuple[int, int], trainable: bool) -> Model:
    backbone = tf.keras.applications.DenseNet201(
        include_top=False,
        weights="imagenet",
        input_shape=(*image_size, 3),
        pooling=None,
    )
    backbone.trainable = trainable
    return backbone


def build_corrected_benchmark_model(config: ExperimentConfig) -> Model:
    """
    Corrected benchmark version.

    The uploaded baseline computes self-attention on pool3 features but then discards it.
    This implementation actually fuses the attention branch into the forward path.
    """
    backbone = _make_backbone(config.image_size, trainable=not config.freeze_backbone)

    pool3 = backbone.get_layer("pool3_relu").output
    pool3_att = AttentionBlock(attention_type="self", name="benchmark_self_attention")(pool3)
    pool3_down = layers.AveragePooling2D(pool_size=2, name="benchmark_pool3_downsample")(pool3_att)

    conv5 = backbone.get_layer("conv5_block32_concat").output
    pool3_proj = layers.Conv2D(
        filters=int(conv5.shape[-1]), kernel_size=1, padding="same", name="benchmark_pool3_projection"
    )(pool3_down)

    fused = layers.Concatenate(name="benchmark_fused_features")([conv5, pool3_proj])
    fused = layers.Conv2D(1024, kernel_size=1, activation="relu", padding="same", name="benchmark_fusion_conv")(fused)
    gap = layers.GlobalAveragePooling2D(name="benchmark_gap")(fused)
    x = layers.Dense(512, activation="relu", kernel_regularizer=l2(config.weight_decay), name="benchmark_fc1")(gap)
    x = layers.Dropout(0.5, name="benchmark_dropout")(x)
    x = layers.BatchNormalization(name="benchmark_bn")(x)
    x = layers.Dense(128, activation="relu", name="benchmark_fc2")(x)
    outputs = layers.Dense(len(TMD_LABELS), activation="softmax", name="tmd_output")(x)

    return Model(inputs=backbone.input, outputs=outputs, name="DenseNet201_Benchmark_Corrected")



def build_agml_model(config: ExperimentConfig) -> Model:
    """
    Proposed thesis model:
    DenseNet201 backbone + CBAM + shared representation + dual heads.
    """
    backbone = _make_backbone(config.image_size, trainable=not config.freeze_backbone)

    conv5 = backbone.get_layer("conv5_block32_concat").output
    attended = AttentionBlock(attention_type="cbam", name="cbam_attention")(conv5)
    attended = layers.Conv2D(1024, kernel_size=1, activation="relu", padding="same", name="cbam_refine_conv")(attended)

    shared = layers.GlobalAveragePooling2D(name="shared_gap")(attended)
    shared = layers.Dense(512, activation="relu", kernel_regularizer=l2(config.weight_decay), name="shared_fc1")(shared)
    shared = layers.Dropout(0.5, name="shared_dropout")(shared)
    shared = layers.BatchNormalization(name="shared_bn")(shared)
    shared = layers.Dense(128, activation="relu", name="shared_fc2")(shared)

    tmd_output = layers.Dense(len(TMD_LABELS), activation="softmax", name="tmd_output")(shared)
    artifact_output = layers.Dense(
        len(ARTIFACT_LABELS), activation="softmax", name="artifact_output"
    )(shared)

    return Model(inputs=backbone.input, outputs=[tmd_output, artifact_output], name="DenseNet201_CBAM_AGML")


In [55]:
# Training helpers

def make_callbacks() -> List[tf.keras.callbacks.Callback]:
    return [
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-6, verbose=1),
    ]



def compile_model(model: Model, multi_task: bool, config: ExperimentConfig) -> None:
    if multi_task:
        model.compile(
            optimizer=Adam(config.learning_rate),
            loss={
                "tmd_output": "categorical_crossentropy",
                "artifact_output": "categorical_crossentropy",
            },
            loss_weights={
                "tmd_output": config.tmd_loss_weight,
                "artifact_output": config.artifact_loss_weight,
            },
            metrics={
                "tmd_output": ["accuracy"],
                "artifact_output": ["accuracy"],
            },
            jit_compile=False,  # Disable JIT for better compatibility across TF versions
        )
    else:
        model.compile(
            optimizer=Adam(config.learning_rate),
            loss="categorical_crossentropy",
            metrics=["accuracy"],
            jit_compile=False,  # Disable JIT for better compatibility across TF versions   
        )



def split_train_val(train_df: pd.DataFrame, val_size: float, random_state: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_part, val_part = train_test_split(
        train_df,
        test_size=val_size,
        stratify=train_df["tmd_label"],
        random_state=random_state,
    )
    return train_part.reset_index(drop=True), val_part.reset_index(drop=True)

In [56]:
# Metrics and evaluation

def compute_binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    accuracy = (tp + tn) / max(cm.sum(), 1)
    specificity = tn / max(tn + fp, 1)
    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

def _collect_predictions(model: Model, generator: TMJSequence, multi_task: bool) -> Tuple[np.ndarray, np.ndarray]:
    y_true: List[int] = []
    y_pred: List[int] = []

    for batch_x, batch_y in generator:
        preds = model.predict(batch_x, verbose=0)
        if multi_task:
            pred_tmd = preds[0]
            true_tmd = batch_y["tmd_output"]
        else:
            pred_tmd = preds
            true_tmd = batch_y

        y_true.extend(np.argmax(true_tmd, axis=1).tolist())
        y_pred.extend(np.argmax(pred_tmd, axis=1).tolist())

    return np.array(y_true), np.array(y_pred)



def measure_inference_speed(model: Model, generator: TMJSequence, warmup_batches: int = 1) -> float:
    batches = list(range(len(generator)))
    if not batches:
        raise ValueError("Generator is empty; cannot measure inference speed.")

    for i in range(min(warmup_batches, len(generator))):
        batch_x, _ = generator[i]
        model.predict(batch_x, verbose=0)

    total_images = 0
    start = time.perf_counter()
    for i in batches:
        batch_x, _ = generator[i]
        _ = model.predict(batch_x, verbose=0)
        total_images += len(batch_x)
    elapsed = time.perf_counter() - start

    return total_images / max(elapsed, 1e-8)



def summarize_cv_results(results_df: pd.DataFrame) -> pd.DataFrame:
    metric_cols = [
        "accuracy",
        "precision",
        "recall",
        "specificity",
        "f1",
        "images_per_second",
    ]
    summary = results_df[metric_cols].agg(["mean", "std"]).T
    summary.columns = ["mean", "std"]
    return summary.reset_index(names="metric")

In [57]:
# Fixed-split experiment runner (used for each generated fold)

def run_fixed_split_experiment(
    config: ExperimentConfig,
    model_type: str = "proposed",
    scenario: str = "clean",
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Uses the existing folder splits directly:

    dataset_root/
        train/normal, train/subluxation
        validation/normal, validation/subluxation
        test/normal, test/subluxation
    """
    if model_type not in {"benchmark", "proposed"}:
        raise ValueError("model_type must be 'benchmark' or 'proposed'")
    if scenario not in {"clean", "artifact_mix"}:
        raise ValueError("scenario must be 'clean' or 'artifact_mix'")

    seed_everything(config.random_state)
    df = index_dataset(config.dataset_root)

    required_splits = {"train", "validation", "test"}
    available_splits = set(df["split"].unique().tolist())
    if not required_splits.issubset(available_splits):
        raise ValueError(
            "Fixed-split experiment requires dataset_root/train, dataset_root/validation, and dataset_root/test."
        )

    train_df = df[df["split"] == "train"].reset_index(drop=True)
    val_df = df[df["split"] == "validation"].reset_index(drop=True)
    test_df = df[df["split"] == "test"].reset_index(drop=True)

    multi_task = model_type == "proposed"
    train_gen = TMJSequence(
        train_df,
        image_size=config.image_size,
        batch_size=config.batch_size,
        multi_task=multi_task,
        scenario=scenario,
        training=True,
        seed=config.random_state,
    )
    val_gen = TMJSequence(
        val_df,
        image_size=config.image_size,
        batch_size=config.batch_size,
        multi_task=multi_task,
        scenario=scenario,
        training=False,
        seed=config.random_state,
    )
    test_gen = TMJSequence(
        test_df,
        image_size=config.image_size,
        batch_size=config.batch_size,
        multi_task=multi_task,
        scenario=scenario,
        training=False,
        seed=config.random_state,
    )

    if model_type == "benchmark":
        model = build_corrected_benchmark_model(config)
    else:
        model = build_agml_model(config)
    compile_model(model, multi_task=multi_task, config=config)

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=config.epochs,
        callbacks=make_callbacks(),
        verbose=1,
    )

    y_true, y_pred = _collect_predictions(model, test_gen, multi_task=multi_task)
    metrics = compute_binary_metrics(y_true, y_pred)
    images_per_second = measure_inference_speed(model, test_gen)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    results_df = pd.DataFrame(
        [
            {
                "split_mode": "fixed",
                "model_type": model_type,
                "scenario": scenario,
                **metrics,
                "images_per_second": float(images_per_second),
                "epochs_ran": len(history.history.get("loss", [])),
            }
        ]
    )
    cm_df = pd.DataFrame(cm, index=DISPLAY_LABELS, columns=DISPLAY_LABELS)
    return results_df, cm_df


In [58]:
# Grad-CAM

def _find_target_layer(model: Model, preferred: Optional[str] = None) -> str:
    if preferred is not None:
        return preferred
    for candidate in ["cbam_refine_conv", "benchmark_fusion_conv", "conv5_block32_concat"]:
        try:
            model.get_layer(candidate)
            return candidate
        except ValueError:
            continue
    raise ValueError("No suitable Grad-CAM target layer found.")



def make_gradcam_heatmap(
    model: Model,
    image_array: np.ndarray,
    target_class: Optional[int] = None,
    target_layer: Optional[str] = None,
    output_name: str = "tmd_output",
) -> np.ndarray:
    target_layer = _find_target_layer(model, target_layer)

    if image_array.ndim == 3:
        image_batch = np.expand_dims(image_array, axis=0)
    else:
        image_batch = image_array

    conv_layer = model.get_layer(target_layer)
    grad_model = Model(
        inputs=model.inputs,
        outputs=[conv_layer.output, model.get_layer(output_name).output],
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(image_batch)
        if target_class is None:
            target_class = int(tf.argmax(predictions[0]))
        class_channel = predictions[:, target_class]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0) / tf.maximum(tf.reduce_max(heatmap), 1e-8)
    return heatmap.numpy()



def overlay_gradcam(
    rgb_image: np.ndarray,
    heatmap: np.ndarray,
    alpha: float = 0.35,
) -> np.ndarray:
    image = rgb_image.copy()
    if image.max() <= 1.0:
        image = (image * 255).astype(np.uint8)
    heatmap = cv2.resize(heatmap, (image.shape[1], image.shape[0]))
    heatmap_uint8 = np.uint8(255 * heatmap)
    color_map = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    color_map = cv2.cvtColor(color_map, cv2.COLOR_BGR2RGB)
    blended = cv2.addWeighted(image, 1.0 - alpha, color_map, alpha, 0)
    return blended

In [59]:
# Plotting helpers

def plot_confusion_matrix(cm_df: pd.DataFrame, title: str = "Confusion Matrix") -> None:
    plt.figure(figsize=(6, 5))
    plt.imshow(cm_df.values, cmap="Blues")
    plt.title(title)
    plt.xticks(range(len(cm_df.columns)), cm_df.columns)
    plt.yticks(range(len(cm_df.index)), cm_df.index)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    for i in range(cm_df.shape[0]):
        for j in range(cm_df.shape[1]):
            plt.text(j, i, int(cm_df.iloc[i, j]), ha="center", va="center")

    plt.tight_layout()
    plt.show()

## 5-Fold training setup

Set this to the folder created by `make_5fold_dataset.py`.


In [ ]:
# 5-fold settings

# FOLDS_ROOT is set in the Colab / Local Dataset Setup section above.
# If you skipped that section locally, this fallback keeps the old behavior.
if "FOLDS_ROOT" not in globals():
    FOLDS_ROOT = Path("data_5_fold")

MODEL_TYPE = "proposed"      # "proposed" or "benchmark"
SCENARIO = "artifact_mix"    # "clean" or "artifact_mix"

# Colab/GPU defaults.
# For a free Colab GPU, start with 4. If you get OOM, use 2.
BATCH_SIZE = 4
EPOCHS = 30
LEARNING_RATE = 1e-4
IMAGE_SIZE = (224, 224)

OUTPUT_DIR = Path("results_5fold_proposed_artifact")

print("FOLDS_ROOT =", Path(FOLDS_ROOT).resolve())
print("OUTPUT_DIR =", OUTPUT_DIR.resolve())


FOLDS_ROOT = /media/solyviemiyu/Drive/Programming/Thesis/AGMTL-DenseCBAM/data_5_fold
OUTPUT_DIR = /media/solyviemiyu/Drive/Programming/Thesis/AGMTL-DenseCBAM/results_5fold_proposed_artifact


In [61]:
# Helper to list fold folders

def find_fold_dirs(folds_root: Path) -> List[Path]:
    if not folds_root.exists():
        raise FileNotFoundError(f"Fold root not found: {folds_root}")

    fold_dirs = sorted(
        [p for p in folds_root.iterdir() if p.is_dir() and p.name.startswith("fold_")],
        key=lambda p: int(p.name.split("_")[1]),
    )
    if not fold_dirs:
        raise FileNotFoundError(
            f"No fold folders found under {folds_root}. Expected fold_1, fold_2, ..."
        )
    return fold_dirs


fold_dirs = find_fold_dirs(FOLDS_ROOT)
[p.name for p in fold_dirs]


['fold_1', 'fold_2', 'fold_3', 'fold_4', 'fold_5']

In [62]:
# Optional: inspect one fold quickly

def summarize_fold(fold_root: Path) -> pd.DataFrame:
    rows = []
    for split_name in ["train", "validation", "test"]:
        for class_name in TMD_LABELS:
            class_dir = fold_root / split_name / class_name
            count = sum(
                1 for p in class_dir.rglob("*")
                if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
            )
            rows.append({
                "fold": fold_root.name,
                "split": split_name,
                "class_name": class_name,
                "count": count,
            })
    return pd.DataFrame(rows)

summarize_fold(fold_dirs[0])


,fold,split,class_name,count
0,fold_1,train,normal,1005
1,fold_1,train,subluxation,1324
2,fold_1,validation,normal,177
3,fold_1,validation,subluxation,234
4,fold_1,test,normal,296
5,fold_1,test,subluxation,389


## Test one fold first

Run this before the full 5-fold sequence.


In [ ]:
TEST_FOLD = fold_dirs[0]   # fold_1

cfg = ExperimentConfig(
    dataset_root=str(TEST_FOLD),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
)

results_one, cm_one = run_fixed_split_experiment(
    cfg,
    model_type=MODEL_TYPE,
    scenario=SCENARIO,
)

print(results_one)
print(cm_one)

plot_confusion_matrix(cm_one, title=f"{MODEL_TYPE} - {SCENARIO} - {TEST_FOLD.name}")


## Sequential 5-fold training

This trains **one fold at a time**, saves each fold result, clears memory, then continues.


In [64]:
METRIC_COLUMNS = [
    "accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "images_per_second",
    "epochs_ran",
]

def run_all_folds(
    folds_root: Path,
    output_dir: Path,
    model_type: str,
    scenario: str,
    image_size: Tuple[int, int],
    batch_size: int,
    epochs: int,
    learning_rate: float,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    output_dir.mkdir(parents=True, exist_ok=True)
    fold_dirs = find_fold_dirs(folds_root)

    rows: List[Dict[str, float]] = []
    all_confusions: List[pd.DataFrame] = []

    for fold_dir in fold_dirs:
        print(f"\n=== Running {fold_dir.name} | model={model_type} | scenario={scenario} ===")

        cfg = ExperimentConfig(
            dataset_root=str(fold_dir),
            image_size=image_size,
            batch_size=batch_size,
            epochs=epochs,
            learning_rate=learning_rate,
        )

        results_df, cm_df = run_fixed_split_experiment(
            cfg,
            model_type=model_type,
            scenario=scenario,
        )

        results_df = results_df.copy()
        results_df["fold"] = fold_dir.name

        results_df.to_csv(output_dir / f"{fold_dir.name}_results.csv", index=False)
        cm_df.to_csv(output_dir / f"{fold_dir.name}_confusion_matrix.csv")

        rows.extend(results_df.to_dict(orient="records"))
        all_confusions.append(cm_df)

        print(results_df)
        print(cm_df)

        tf.keras.backend.clear_session()
        gc.collect()

    combined_df = pd.DataFrame(rows)

    summary_rows = []
    for metric in METRIC_COLUMNS:
        if metric in combined_df.columns:
            summary_rows.append({
                "metric": metric,
                "mean": combined_df[metric].mean(),
                "std": combined_df[metric].std(ddof=1),
            })
    summary_df = pd.DataFrame(summary_rows)

    pooled_cm = all_confusions[0].copy()
    for cm in all_confusions[1:]:
        pooled_cm = pooled_cm.add(cm, fill_value=0)
    pooled_cm = pooled_cm.astype(int)

    combined_df.to_csv(output_dir / "all_fold_results.csv", index=False)
    summary_df.to_csv(output_dir / "summary_mean_std.csv", index=False)
    pooled_cm.to_csv(output_dir / "pooled_confusion_matrix.csv")

    metadata = {
        "folds_root": str(folds_root.resolve()),
        "output_dir": str(output_dir.resolve()),
        "model_type": model_type,
        "scenario": scenario,
        "image_size": list(image_size),
        "batch_size": batch_size,
        "epochs": epochs,
        "learning_rate": learning_rate,
    }
    (output_dir / "run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    return combined_df, summary_df, pooled_cm


In [ ]:
all_fold_results, summary_results, pooled_cm = run_all_folds(
    folds_root=FOLDS_ROOT,
    output_dir=OUTPUT_DIR,
    model_type=MODEL_TYPE,
    scenario=SCENARIO,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
)

print("\n=== All fold results ===")
print(all_fold_results)

print("\n=== Mean ± SD summary ===")
print(summary_results)

print("\n=== Pooled confusion matrix ===")
print(pooled_cm)

plot_confusion_matrix(pooled_cm, title=f"{MODEL_TYPE} - {SCENARIO} - Pooled 5-Fold")



=== Running fold_1 | model=proposed | scenario=artifact_mix ===
Epoch 1/30
292/292 ━━━━━━━━━━━━━━━━━━━━ 137s 229ms/step - artifact_output_accuracy: 0.2443 - artifact_output_loss: 0.5138 - loss: 5.5737 - tmd_output_accuracy: 0.2241 - tmd_output_loss: 0.7263 - val_artifact_output_accuracy: 0.3066 - val_artifact_output_loss: 0.9627 - val_loss: 4.1841 - val_tmd_output_accuracy: 0.3431 - val_tmd_output_loss: 0.6701 - learning_rate: 1.0000e-04
Epoch 2/30
292/292 ━━━━━━━━━━━━━━━━━━━━ 63s 215ms/step - artifact_output_accuracy: 0.2525 - artifact_output_loss: 0.2058 - loss: 3.0919 - tmd_output_accuracy: 0.2477 - tmd_output_loss: 0.6014 - val_artifact_output_accuracy: 0.2482 - val_artifact_output_loss: 0.1806 - val_loss: 2.6079 - val_tmd_output_accuracy: 0.1411 - val_tmd_output_loss: 0.6933 - learning_rate: 1.0000e-04
Epoch 3/30
292/292 ━━━━━━━━━━━━━━━━━━━━ 62s 213ms/step - artifact_output_accuracy: 0.2538 - artifact_output_loss: 0.1437 - loss: 2.0484 - tmd_output_accuracy: 0.2525 - tmd_output_l

## Download results from Colab

After training finishes, run this cell to zip the results folder and download it.


In [ ]:
def zip_results_folder(output_dir: Path) -> Path:
    output_dir = Path(output_dir)
    zip_base = output_dir.with_suffix("")
    zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=output_dir)
    return Path(zip_path)


results_zip = zip_results_folder(OUTPUT_DIR)
print("Created:", results_zip)

if IN_COLAB:
    files.download(str(results_zip))
else:
    print("Local result zip:", results_zip.resolve())


## Suggested thesis runs

Change only `MODEL_TYPE`, `SCENARIO`, and `OUTPUT_DIR`, then rerun the last two cells.

### Proposed + clean
```python
MODEL_TYPE = "proposed"
SCENARIO = "clean"
OUTPUT_DIR = Path("results_prop_clean")
```

### Proposed + artifact_mix
```python
MODEL_TYPE = "proposed"
SCENARIO = "artifact_mix"
OUTPUT_DIR = Path("results_prop_artifact")
```

### Benchmark + clean
```python
MODEL_TYPE = "benchmark"
SCENARIO = "clean"
OUTPUT_DIR = Path("results_bench_clean")
```

### Benchmark + artifact_mix
```python
MODEL_TYPE = "benchmark"
SCENARIO = "artifact_mix"
OUTPUT_DIR = Path("results_bench_artifact")
```
